# Занятие 9. ООП: объект как модель предметной области

**План занятия**

1. Боль, из которой выросло ООП
2. Класс и объект
3. Зона ответственности
4. Инкапсуляция
5. Композиция и наследование
6. Когда класс не нужен
7. Домашние задачи

**Теория:** `theory/09_ООП.md`
Т. Гэддис, гл. 10 (с. 528 / PDF 553)

In [ ]:
# Эта ячейка находит папку с данными. Запустите её ПЕРВОЙ.
import os

CANDIDATES = ["../data", "data", "./data", "/content/data",
              "/content/drive/MyDrive/mglu/data"]
DATA = next((p for p in CANDIDATES if os.path.isdir(p)), None)
print("Data folder:", os.path.abspath(DATA) if DATA else "NOT FOUND")

---

## 1. Боль, из которой всё выросло

Данные в словаре, функции рядом, связь между ними держится в голове.

In [ ]:
doc = {"id": "doc01", "text": "One two three two", "tokens": None}

def tokenize(document):
    document["tokens"] = document["text"].lower().split()

def word_count(document):
    return len(document["tokens"])

tokenize(doc)
print(word_count(doc))

In [ ]:
# что будет, если забыть про порядок вызовов
fresh = {"id": "doc02", "text": "One two", "tokens": None}

try:
    print(word_count(fresh))
except TypeError as error:
    print("TypeError:", error)
    print("word_count требует, чтобы кто-то заранее вызвал tokenize")

Вопросы, на которые такой код не отвечает. Можно ли вызвать `word_count`
до `tokenize`? Что случится при повторном вызове? Кто отвечает за согласованность
полей?

---

## 2. Класс и объект

In [ ]:
class Document:
    def __init__(self, doc_id, text):
        self.doc_id = doc_id
        self.text = text
        self.tokens = None

    def tokenize(self):
        self.tokens = self.text.lower().split()
        return self.tokens

    def word_count(self):
        if self.tokens is None:      # метод сам следит за состоянием
            self.tokenize()
        return len(self.tokens)


doc = Document("doc01", "One two three two")
print(doc.doc_id)
print(doc.word_count())      # tokenize вызывать заранее не нужно
print(doc.tokens)

`__init__` вызывается при создании объекта. `self` это ссылка на сам объект.
В списке параметров `self` пишут, при вызове не пишут.

**Проверьте себя.** Что напечатает следующая ячейка?

In [ ]:
a = Document("doc01", "one two")
b = Document("doc02", "three four five")

print(a.word_count(), b.word_count())
print(a.tokens, b.tokens)

У каждого объекта своё состояние. Это главное отличие от модуля с функциями.

---

## 3. Согласованность состояния

Если поле `tokens` заполнено, оно обязано соответствовать полю `text`.

In [ ]:
doc = Document("doc01", "one two")
print(doc.word_count())

doc.text = "one two three four five"     # поменяли текст напрямую
print(doc.word_count(), "устаревший ответ, tokens не пересчитались")

In [ ]:
class Document:
    def __init__(self, doc_id, text):
        self.doc_id = doc_id
        self._text = text
        self._tokens = None

    def get_text(self):
        return self._text

    def set_text(self, text):
        self._text = text
        self._tokens = None          # старые токены больше не действительны

    def tokenize(self):
        self._tokens = self._text.lower().split()
        return self._tokens

    def word_count(self):
        if self._tokens is None:
            self.tokenize()
        return len(self._tokens)

    def __str__(self):
        return f"Document {self.doc_id!r} ({self.word_count()} words)"


doc = Document("doc01", "one two")
print(doc.word_count())
doc.set_text("one two three four five")
print(doc.word_count(), "теперь верно")
print(doc)

Подчёркивание в начале имени это соглашение: поле внутреннее.
Python это не проверяет, договорённость держится на дисциплине.

Метод `__str__` задаёт текстовое представление. Без него `print` выводит
адрес объекта в памяти.

---

## 4. Зона ответственности

У класса должна быть одна зона ответственности, называемая одной фразой
без слова «и».

In [ ]:
class Tokenizer:
    """Отвечает за правила разбиения текста на токены."""

    def __init__(self, lowercase=True, keep_hyphens=False):
        self._lowercase = lowercase
        self._keep_hyphens = keep_hyphens

    def split(self, text):
        if self._lowercase:
            text = text.lower()
        separators = " .,!?:()"
        if not self._keep_hyphens:
            separators += "-"
        for ch in separators:
            text = text.replace(ch, " ")
        return [t for t in text.split() if t]


sample = "Кто-то сказал: «Это текст!»"

plain = Tokenizer()
hyphens = Tokenizer(keep_hyphens=True)

print(plain.split(sample))
print(hyphens.split(sample))

Настройки токенизации стали состоянием объекта. Вызывающий код больше
не передаёт их в каждый вызов, и правило применяется единообразно.

---

## 5. Композиция

Корпус содержит токенизатор и документы.

In [ ]:
import glob

class Corpus:
    """Отвечает за набор документов и операции над набором."""

    def __init__(self, tokenizer):
        self._tokenizer = tokenizer      # СОДЕРЖИТ токенизатор
        self._documents = []

    def add(self, document):
        self._documents.append(document)
        return self

    def __len__(self):
        return len(self._documents)

    def vocabulary(self):
        words = set()
        for document in self._documents:
            words.update(self._tokenizer.split(document.get_text()))
        return words

    def total_words(self):
        return sum(len(self._tokenizer.split(d.get_text()))
                   for d in self._documents)

    def __str__(self):
        return f"Corpus of {len(self)} documents, {len(self.vocabulary())} types"


corpus = Corpus(Tokenizer())
for path in sorted(glob.glob(f"{DATA}/corpus/*.txt")):
    with open(path, encoding="utf-8") as f:
        corpus.add(Document(os.path.basename(path), f.read()))

print(corpus)
print("всего словоупотреблений:", corpus.total_words())

In [ ]:
# заменить правило токенизации означает поменять один объект
strict = Corpus(Tokenizer(keep_hyphens=True))
for path in sorted(glob.glob(f"{DATA}/corpus/*.txt")):
    with open(path, encoding="utf-8") as f:
        strict.add(Document(os.path.basename(path), f.read()))

plain_size = len(corpus.vocabulary())
strict_size = len(strict.vocabulary())

print("дефис как разделитель:", plain_size, "типов")
print("дефис внутри слова:   ", strict_size, "типов")
print()
print(f"словарь изменился на {strict_size - plain_size}: слова вроде «кто-то»")
print("перестали распадаться на два токена")

Классы `Document` и `Corpus` не поменялись. Это и есть проверка качества
разбиения из занятия 10.

---

## 6. Наследование

Правило проверки: подставьте фразу «X является разновидностью Y».

In [ ]:
import re

class XmlDocument(Document):
    """Документ с разметкой. XmlDocument является разновидностью Document."""

    def get_text(self):
        return re.sub(r"<[^>]+>", " ", self._text)


marked = XmlDocument("doc_xml", "<p>One <b>two</b> three</p>")
print(repr(marked.get_text()))
print(marked.word_count())

| Фраза | Звучит | Решение |
| --- | --- | --- |
| XmlDocument является разновидностью Document | да | наследование |
| Corpus является разновидностью Document | нет | композиция |
| Tokenizer является разновидностью Corpus | нет | композиция |

Наследованием злоупотребляют, потому что оно даёт быстрый способ переиспользовать
код. Композиция для этой цели подходит лучше и связывает классы слабее.

---

## 7. Когда класс не нужен

In [ ]:
# функция без состояния остаётся функцией
def average_length(words):
    return sum(len(w) for w in words) / len(words) if words else 0.0

print(average_length(["one", "three", "five"]))

In [ ]:
# данные без поведения: достаточно NamedTuple
from typing import NamedTuple

class Match(NamedTuple):
    doc_id: str
    position: int
    fragment: str

m = Match("doc01", 42, "one two three")
print(m)
print(m.doc_id, m.position)
print("распаковывается как кортеж:", tuple(m))

| Ситуация | Решение |
| --- | --- |
| функция без состояния | обычная функция |
| данные без поведения | `NamedTuple` или `dataclass` |
| один экземпляр на программу | модуль с функциями |
| класс из одного метода | функция |

Класс оправдан там, где есть состояние, требующее согласованности,
и несколько операций над ним.

---

# Домашние задачи

Рассчитаны примерно на 30 минут.

### Задача 1. Трассировка (без запуска)

Что напечатает код и почему? Ловушка та же, что на занятии 3.

In [ ]:
# Мой ответ: ...

# class Basket:
#     def __init__(self, items=[]):
#         self.items = items
#     def add(self, item):
#         self.items.append(item)
#         return self.items
#
# a = Basket()
# b = Basket()
# a.add("x")
# print(b.add("y"))

### Задача 2. Минимум

Спроектируйте **на бумаге** классы для выбранной предметной области.
Для каждого класса запишите:

1. зону ответственности одной фразой без слова «и»,
2. поля,
3. методы,
4. что снаружи и что внутри.

Код писать не требуется. Проектирование это и есть содержание задания.

### Задача 3. Класс `Concordance`

Напишите класс, который принимает корпус и по запросу выдаёт все вхождения слова
с контекстом заданной ширины. Зона ответственности одна, назовите её.

In [ ]:
class Concordance:
    def __init__(self, corpus, width=5):
        # ваш код здесь
        pass

    def find(self, word):
        """Возвращает список строк с контекстом вокруг вхождений слова."""
        pass

# c = Concordance(corpus)
# for line in c.find("язык")[:5]:
#     print(line)

### Задача 4. Найдите нарушение

В классе ниже три проблемы из раздела 4. Найдите их и скажите,
на сколько классов его следует разделить.

In [ ]:
# class TextProcessor:
#     def __init__(self, path):
#         self.path = path
#         self.text = None
#         self.tokens = None
#         self.counts = None
#
#     def load(self): ...
#     def tokenize(self): ...
#     def count(self): ...
#     def save_report(self, out_path): ...
#     def send_email(self, address): ...
#     @staticmethod
#     def is_valid_email(address): ...

# Проблема 1:
# Проблема 2:
# Проблема 3:
# Сколько классов нужно:

### Задача 5. Подумать (кода не нужно)

Вы добавляете в `Corpus` метод `search(word)`. Он относится к зоне
ответственности корпуса? Обоснуйте оба варианта ответа, потом выберите один.

---

# Итоги

- Класс складывает данные и работающий с ними код в одно место.
- `__init__` задаёт начальное состояние, `self` это ссылка на объект.
- У каждого объекта своё состояние.
- Зона ответственности называется одной фразой без слова «и».
- Метод, не обращающийся к полям, это функция.
- Композиция это «содержит», наследование это «является разновидностью».
- Изменяемое значение по умолчанию в `__init__` разделяется всеми объектами.
- Функция из трёх строк не становится лучше от обёртывания в класс.